# Member B — SIFT + Bag-of-Visual-Words

This notebook is the **Member B** entry point. It covers the full traditional pipeline that previously lived in CLI scripts:

1. Run SIFT → BoVW → SVM / Random Forest (`src.traditional.run_pipeline`)
2. Compare vocabulary-size ablations (was `scripts/compare_vocab_ablation.py`)
3. Export shared-evaluation NPZ files for Member A (was `scripts/export_shared_eval_scores.py`)
4. Export success / failure case shortlists (was `scripts/export_prediction_cases.py`)
5. Optionally re-check metrics with Member A's shared evaluator

Core implementation stays in `src/traditional/` so the NPZ schema and protocol already handed to Member A remain valid. Retraining switches default to **off** to avoid accidental multi-hour runs.


## 0. Environment and run switches

Use the **9517** Jupyter kernel. First-time setup:

```bash
pip install -r requirements.txt
# or use the project .venv
```


In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# --- project root: cwd, else walk parents for src/traditional ---
def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "traditional").is_dir() and (candidate / "data_metadata").is_dir():
            return candidate
    raise FileNotFoundError(
        "Cannot find project root (need src/traditional and data_metadata). "
        "Open this notebook from the 9517_Proj repository."
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.traditional.classifiers import decision_scores
from src.traditional.common import load_yaml
from src.traditional.run_pipeline import run_pipeline
from src.evaluation.evaluate_predictions import evaluate_scores, load_npz, save_evaluation

print("PROJECT_ROOT =", PROJECT_ROOT)

# ========== RUN SWITCHES (safe defaults) ==========
# Full SIFT + encode on 500 classes is slow; keep False unless you intend to retrain.
RUN_PILOT = False          # 20-class smoke test
RUN_FULL_K1000 = False     # baseline K=1000 (+ RF if in config)
RUN_ABLATE_K500 = False
RUN_ABLATE_K2000 = False   # selected final model
RUN_RF_LIGHT = False       # light RF resume from features
FROM_FEATURES = True       # resume classifiers from features.npz when available

# Post-processing (cheap; OK to leave True when outputs/ already exist)
RUN_COMPARE_ABLATION = True
RUN_EXPORT_SHARED_EVAL = True
RUN_EXPORT_CASES = True
RUN_A_EVALUATOR_CHECK = True


PROJECT_ROOT = /Users/peanut/Code/UNSW/COMP9517/9517_Proj


## 1. Method summary

- **Preprocess:** grayscale; if longer side > 512, resize with `INTER_AREA` so `max(h, w) = 512`
- **SIFT:** OpenCV, `nfeatures=0`, `contrastThreshold=0.04`
- **Vocabulary:** train split only; up to 500k descriptors; `MiniBatchKMeans`, seed **9517**
- **Encoding:** hard-assignment BoVW → TF–IDF (fit on train) → L2
- **Classifiers:** LinearSVC (final) / RandomForest (control)
- **Protocol:** choose $K$ / hyperparameters on **validation only**; after selection, evaluate **test once**


## 2. Run the pipeline

Equivalent CLI:

```bash
python -m src.traditional.run_pipeline --config ...
```


In [2]:
def run_config(rel_config: str, *, from_features: bool = False) -> Path:
    config_path = PROJECT_ROOT / rel_config
    if not config_path.exists():
        raise FileNotFoundError(config_path)
    config = load_yaml(config_path)
    print(f"\n=== Running {rel_config} | from_features={from_features} ===", flush=True)
    out = run_pipeline(config, PROJECT_ROOT, from_features=from_features)
    print(f"Done → {out}", flush=True)
    return out


def maybe_from_features(rel_config: str) -> bool:
    '''Use cached features.npz when present and FROM_FEATURES is True.'''
    if not FROM_FEATURES:
        return False
    cfg = load_yaml(PROJECT_ROOT / rel_config)
    feats = PROJECT_ROOT / cfg["experiment"]["output_dir"] / "features.npz"
    return feats.exists()


jobs = []
if RUN_PILOT:
    jobs.append("configs/b0_pilot_20cls.yaml")
if RUN_FULL_K1000:
    jobs.append("configs/b1_full500_k1000.yaml")
if RUN_ABLATE_K500:
    jobs.append("configs/b1_ablate_k500.yaml")
if RUN_ABLATE_K2000:
    jobs.append("configs/b1_ablate_k2000.yaml")
if RUN_RF_LIGHT:
    jobs.append("configs/b1_rf_only_light.yaml")

if not jobs:
    print("No training jobs enabled. Toggle RUN_* switches above to retrain.")
else:
    for rel in jobs:
        run_config(rel, from_features=maybe_from_features(rel))


No training jobs enabled. Toggle RUN_* switches above to retrain.


## 3. Vocabulary-size ablation

Equivalent to `scripts/compare_vocab_ablation.py`.

Select the final $K$ by **validation macro-F1** (break ties with Top-1).


In [3]:
DEFAULT_RUNS = [
    ("K=500", "outputs/b1_ablate_k500"),
    ("K=1000", "outputs/b1_full500_sift_bovw_k1000"),
    ("K=2000", "outputs/b1_ablate_k2000"),
]


def load_metrics(run_dir: Path, split: str) -> dict | None:
    path = run_dir / "svm" / f"metrics_{split}.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def compare_vocab_ablation(runs=DEFAULT_RUNS, out_csv: Path | None = None) -> pd.DataFrame:
    out_csv = out_csv or (PROJECT_ROOT / "outputs" / "ablation_vocabulary_svm.csv")
    rows = []
    print(f"{'run':<10} {'split':<5} {'top1':>8} {'top5':>8} {'macro_f1':>10}")
    print("-" * 48)
    for label, rel in runs:
        run_dir = PROJECT_ROOT / rel
        for split in ("val", "test"):
            metrics = load_metrics(run_dir, split)
            if metrics is None:
                print(f"{label:<10} {split:<5} {'MISSING':>8}")
                continue
            row = {
                "run": label,
                "output_dir": rel,
                "split": split,
                "top1_accuracy": metrics["top1_accuracy"],
                "top5_accuracy": metrics["top5_accuracy"],
                "macro_precision": metrics["macro_precision"],
                "macro_recall": metrics["macro_recall"],
                "macro_f1": metrics["macro_f1"],
            }
            rows.append(row)
            print(
                f"{label:<10} {split:<5} "
                f"{metrics['top1_accuracy']*100:7.2f}% "
                f"{metrics['top5_accuracy']*100:7.2f}% "
                f"{metrics['macro_f1']*100:9.2f}%"
            )
    df = pd.DataFrame(rows)
    if len(df):
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_csv, index=False)
        print(f"\nWrote {out_csv}")
        val_rows = df[df["split"] == "val"]
        if len(val_rows):
            best = val_rows.sort_values(["macro_f1", "top1_accuracy"], ascending=False).iloc[0]
            print(
                f"\nBest on validation by macro-F1: {best['run']} "
                f"(top1={best['top1_accuracy']*100:.2f}%, "
                f"macro_f1={best['macro_f1']*100:.2f}%)"
            )
    return df


ablation_df = compare_vocab_ablation() if RUN_COMPARE_ABLATION else None
ablation_df


run        split     top1     top5   macro_f1
------------------------------------------------
K=500      val      8.28%   18.72%      6.54%
K=500      test     8.82%   18.68%      7.07%
K=1000     val      8.66%   19.18%      7.47%
K=1000     test     8.80%   18.82%      7.46%
K=2000     val      9.08%   18.40%      7.90%
K=2000     test     8.86%   17.80%      7.71%

Wrote /Users/peanut/Code/UNSW/COMP9517/9517_Proj/outputs/ablation_vocabulary_svm.csv

Best on validation by macro-F1: K=2000 (top1=9.08%, macro_f1=7.90%)


,run,output_dir,split,top1_accuracy,top5_accuracy,macro_precision,macro_recall,macro_f1
0,K=500,outputs/b1_ablate_k500,val,0.0828,0.1872,0.060053,0.0828,0.065359
1,K=500,outputs/b1_ablate_k500,test,0.0882,0.1868,0.064415,0.0882,0.070651
2,K=1000,outputs/b1_full500_sift_bovw_k1000,val,0.0866,0.1918,0.071365,0.0866,0.074673
3,K=1000,outputs/b1_full500_sift_bovw_k1000,test,0.0880,0.1882,0.070655,0.0880,0.074631
4,K=2000,outputs/b1_ablate_k2000,val,0.0908,0.1840,0.074969,0.0908,0.078979
5,K=2000,outputs/b1_ablate_k2000,test,0.0886,0.1780,0.074148,0.0886,0.077118


## 4. Export shared-evaluation NPZ (for Member A)

Equivalent to `scripts/export_shared_eval_scores.py`.

NPZ schema: `scores (N, 500)`, `true_labels`, `image_paths`, `class_indices`.


In [4]:
def export_shared_eval_scores(
    run_dir: Path,
    *,
    model_name: str,
    metadata_dir: Path | None = None,
    out_dir: Path | None = None,
    splits: tuple[str, ...] = ("val", "test"),
    classifier_subdir: str = "svm",
) -> Path:
    metadata_dir = metadata_dir or (PROJECT_ROOT / "data_metadata")
    out_dir = out_dir or (PROJECT_ROOT / "outputs" / "shared_eval_exports")
    out_dir.mkdir(parents=True, exist_ok=True)

    features = np.load(run_dir / "features.npz")
    model = joblib.load(run_dir / classifier_subdir / "model.joblib")
    timing = json.loads((run_dir / classifier_subdir / "timing.json").read_text())
    training_seconds = float(timing["train_sec"])

    manifest = {
        "model_name": model_name,
        "run_dir": str(run_dir),
        "classifier": classifier_subdir,
        "training_seconds_definition": (
            "Classifier.fit wall time on BoVW training features only "
            "(excludes SIFT extraction, vocabulary, and encoding)."
        ),
        "inference_seconds_definition": (
            "decision_function / ranking score computation wall time on the "
            "split features only (excludes SIFT and BoVW encoding)."
        ),
        "training_seconds": training_seconds,
        "exports": {},
    }

    for split in splits:
        x = features[f"x_{split}"]
        y = features[f"y_{split}"].astype(np.int64)
        frame = pd.read_csv(metadata_dir / f"{split}.csv")
        paths = frame["image_path"].astype(str).to_numpy()
        csv_labels = frame["class_index"].to_numpy(dtype=np.int64)
        if len(paths) != len(x):
            raise ValueError(f"{split}: CSV has {len(paths)} rows but features have {len(x)}")
        if not np.array_equal(y, csv_labels):
            raise ValueError(
                f"{split}: feature labels do not match data_metadata/{split}.csv class_index order"
            )

        scores = decision_scores(model, x).astype(np.float64)
        if scores.shape != (len(x), 500):
            raise ValueError(f"{split}: expected scores shape {(len(x), 500)}, got {scores.shape}")

        timing_split = json.loads(
            (run_dir / classifier_subdir / f"timing_{split}.json").read_text()
        )
        inference_seconds = float(timing_split["inference_sec"])

        out_name = f"{model_name}_{split}_scores.npz"
        out_path = out_dir / out_name
        np.savez_compressed(
            out_path,
            scores=scores,
            true_labels=y.astype(np.int64),
            image_paths=np.asarray(paths, dtype=str),
            class_indices=np.arange(500, dtype=np.int64),
        )
        manifest["exports"][split] = {
            "npz": str(out_path),
            "n_images": int(len(x)),
            "scores_shape": list(scores.shape),
            "inference_seconds": inference_seconds,
            "training_seconds": training_seconds,
        }
        print(
            f"Wrote {out_path}  scores={scores.shape}  "
            f"infer={inference_seconds:.4f}s  train={training_seconds:.1f}s"
        )

    manifest_path = out_dir / f"{model_name}_export_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(f"Wrote {manifest_path}")
    return out_dir


# Exports used in the Member A handoff (skip missing runs gracefully)
EXPORT_SPECS = [
    {
        "run_dir": PROJECT_ROOT / "outputs" / "b1_ablate_k2000",
        "model_name": "member_b_sift_bovw_svm_k2000",
        "splits": ("val", "test"),
        "classifier_subdir": "svm",
    },
    {
        "run_dir": PROJECT_ROOT / "outputs" / "b1_ablate_k500",
        "model_name": "member_b_sift_bovw_svm_k500",
        "splits": ("val",),
        "classifier_subdir": "svm",
    },
    {
        "run_dir": PROJECT_ROOT / "outputs" / "b1_full500_sift_bovw_k1000",
        "model_name": "member_b_sift_bovw_svm_k1000",
        "splits": ("val",),
        "classifier_subdir": "svm",
    },
    {
        "run_dir": PROJECT_ROOT / "outputs" / "b1_full500_sift_bovw_k1000",
        "model_name": "member_b_sift_bovw_rf_light_k1000",
        "splits": ("val",),
        "classifier_subdir": "random_forest",
    },
]

if RUN_EXPORT_SHARED_EVAL:
    for spec in EXPORT_SPECS:
        model_path = spec["run_dir"] / spec["classifier_subdir"] / "model.joblib"
        feats = spec["run_dir"] / "features.npz"
        if not model_path.exists() or not feats.exists():
            print(f"SKIP {spec['model_name']}: missing {model_path.name} or features.npz")
            continue
        export_shared_eval_scores(**spec)
else:
    print("RUN_EXPORT_SHARED_EVAL=False")


Wrote /Users/peanut/Code/UNSW/COMP9517/9517_Proj/outputs/shared_eval_exports/member_b_sift_bovw_svm_k2000_val_scores.npz  scores=(5000, 500)  infer=0.0329s  train=385.9s
Wrote /Users/peanut/Code/UNSW/COMP9517/9517_Proj/outputs/shared_eval_exports/member_b_sift_bovw_svm_k2000_test_scores.npz  scores=(5000, 500)  infer=0.0375s  train=385.9s
Wrote /Users/peanut/Code/UNSW/COMP9517/9517_Proj/outputs/shared_eval_exports/member_b_sift_bovw_svm_k2000_export_manifest.json
Wrote /Users/peanut/Code/UNSW/COMP9517/9517_Proj/outputs/shared_eval_exports/member_b_sift_bovw_svm_k500_val_scores.npz  scores=(5000, 500)  infer=0.0150s  train=184.4s
Wrote /Users/peanut/Code/UNSW/COMP9517/9517_Proj/outputs/shared_eval_exports/member_b_sift_bovw_svm_k500_export_manifest.json
Wrote /Users/peanut/Code/UNSW/COMP9517/9517_Proj/outputs/shared_eval_exports/member_b_sift_bovw_svm_k1000_val_scores.npz  scores=(5000, 500)  infer=0.0459s  train=538.0s
Wrote /Users/peanut/Code/UNSW/COMP9517/9517_Proj/outputs/shared_eva

## 5. Optional: re-check with Member A's evaluator


In [5]:
def check_with_a_evaluator(npz_path: Path, out_dir: Path) -> dict:
    scores, true_labels, class_indices, image_paths = load_npz(npz_path)
    result = evaluate_scores(
        scores,
        true_labels,
        class_indices=class_indices,
        image_paths=image_paths,
    )
    m = result["metrics"]
    print(
        f"{npz_path.name}: top1={m['top1_accuracy']*100:.2f}% "
        f"top5={m['top5_accuracy']*100:.2f}% macro_f1={m['macro_f1']*100:.2f}%"
    )
    save_evaluation(
        result,
        out_dir,
        model_name=npz_path.stem.replace("_scores", ""),
        split="from_npz",
        input_path=npz_path,
        overwrite=True,
    )
    return m


if RUN_A_EVALUATOR_CHECK:
    npz = (
        PROJECT_ROOT
        / "outputs"
        / "shared_eval_exports"
        / "member_b_sift_bovw_svm_k2000_test_scores.npz"
    )
    if npz.exists():
        check_with_a_evaluator(
            npz,
            PROJECT_ROOT / "outputs" / "shared_eval_exports" / "a_evaluator_check_k2000_test",
        )
    else:
        print(f"Missing {npz}; run section 4 first.")
else:
    print("RUN_A_EVALUATOR_CHECK=False")


member_b_sift_bovw_svm_k2000_test_scores.npz: top1=8.86% top5=17.80% macro_f1=7.71%


## 6. Export success / failure case shortlist

Equivalent to `scripts/export_prediction_cases.py`.

Use these cases for the report and the ~2-minute video screenshots.


In [6]:
def species_name(row: pd.Series) -> str:
    for key in ("species", "common_name", "genus"):
        value = row.get(key)
        if value is not None and not (isinstance(value, float) and pd.isna(value)):
            return str(value)
    return f"class_{int(row['class_index'])}"


def export_prediction_cases(
    predictions_csv: Path,
    *,
    metadata_dir: Path | None = None,
    out_dir: Path | None = None,
    n_success: int = 10,
    n_failure: int = 10,
) -> dict:
    metadata_dir = metadata_dir or (PROJECT_ROOT / "data_metadata")
    out_dir = out_dir or (PROJECT_ROOT / "docs" / "member_b" / "cases_k2000_svm")
    out_dir.mkdir(parents=True, exist_ok=True)

    preds = pd.read_csv(predictions_csv).copy()
    tax = pd.read_csv(metadata_dir / "taxonomy.csv").copy()
    tax["class_index"] = tax["class_index"].astype(int)
    name_map = {int(r["class_index"]): species_name(r) for _, r in tax.iterrows()}

    preds["true_name"] = preds["true_class"].map(name_map)
    preds["pred_name"] = preds["predicted_label"].map(name_map)
    preds["confidence"] = preds["confidence"].astype(float)

    correct = preds[preds["is_correct"] == 1].sort_values("confidence", ascending=False)
    wrong = preds[preds["is_correct"] == 0].sort_values("confidence", ascending=False)
    success = correct.head(n_success)
    failure = wrong.head(n_failure)
    mid = wrong.iloc[len(wrong) // 3 : len(wrong) // 3 + max(3, n_failure // 3)]
    failure = (
        pd.concat([failure, mid]).drop_duplicates(subset=["image_id"]).head(n_failure)
    )

    success_path = out_dir / "success_cases.csv"
    failure_path = out_dir / "failure_cases.csv"
    success.to_csv(success_path, index=False)
    failure.to_csv(failure_path, index=False)

    lines = [
        "# Case study shortlist — SIFT-BoVW SVM (K=2000, test)",
        "",
        f"Source: `{predictions_csv}`",
        "",
        "## High-confidence correct predictions",
        "",
        "| # | true | pred | conf | image_path |",
        "|---|---|---|---:|---|",
    ]
    for i, row in enumerate(success.itertuples(index=False), start=1):
        lines.append(
            f"| {i} | {row.true_name} | {row.pred_name} | {row.confidence:.4f} | `{row.image_path}` |"
        )
    lines += [
        "",
        "## High-confidence errors (model is wrong but confident)",
        "",
        "| # | true | pred | conf | image_path |",
        "|---|---|---|---:|---|",
    ]
    for i, row in enumerate(failure.itertuples(index=False), start=1):
        lines.append(
            f"| {i} | {row.true_name} | {row.pred_name} | {row.confidence:.4f} | `{row.image_path}` |"
        )
    lines += [
        "",
        "## How to use",
        "",
        "1. Open paths under `Dataset_v1.0_FROZEN/`.",
        "2. Prefer 3–5 successes and 3–5 failures in the final PDF/video.",
        "",
    ]
    md_path = out_dir / "CASE_SHORTLIST.md"
    md_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    summary = {
        "predictions": str(predictions_csv),
        "n_test": int(len(preds)),
        "n_correct": int((preds["is_correct"] == 1).sum()),
        "n_wrong": int((preds["is_correct"] == 0).sum()),
        "success_csv": str(success_path),
        "failure_csv": str(failure_path),
        "markdown": str(md_path),
    }
    (out_dir / "export_summary.json").write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(summary, indent=2))
    return summary


if RUN_EXPORT_CASES:
    pred_csv = PROJECT_ROOT / "outputs" / "b1_ablate_k2000" / "svm" / "predictions_test.csv"
    if pred_csv.exists():
        export_prediction_cases(pred_csv)
    else:
        print(f"Missing {pred_csv}")
else:
    print("RUN_EXPORT_CASES=False")


{
  "predictions": "/Users/peanut/Code/UNSW/COMP9517/9517_Proj/outputs/b1_ablate_k2000/svm/predictions_test.csv",
  "n_test": 5000,
  "n_correct": 443,
  "n_wrong": 4557,
  "success_csv": "/Users/peanut/Code/UNSW/COMP9517/9517_Proj/docs/member_b/cases_k2000_svm/success_cases.csv",
  "failure_csv": "/Users/peanut/Code/UNSW/COMP9517/9517_Proj/docs/member_b/cases_k2000_svm/failure_cases.csv",
  "markdown": "/Users/peanut/Code/UNSW/COMP9517/9517_Proj/docs/member_b/cases_k2000_svm/CASE_SHORTLIST.md"
}


## 7. Results snapshot (from saved metrics JSON)


In [7]:
def show_summary(run_rel: str, clf: str = "svm") -> None:
    run_dir = PROJECT_ROOT / run_rel
    print(f"\n## {run_rel} / {clf}")
    for split in ("val", "test"):
        p = run_dir / clf / f"metrics_{split}.json"
        if not p.exists():
            print(f"  {split}: MISSING")
            continue
        m = json.loads(p.read_text())
        print(
            f"  {split}: top1={m['top1_accuracy']*100:.2f}% "
            f"top5={m['top5_accuracy']*100:.2f}% "
            f"macro_f1={m['macro_f1']*100:.2f}%"
        )


for rel in [
    "outputs/b1_ablate_k500",
    "outputs/b1_full500_sift_bovw_k1000",
    "outputs/b1_ablate_k2000",
]:
    show_summary(rel, "svm")
show_summary("outputs/b1_full500_sift_bovw_k1000", "random_forest")



## outputs/b1_ablate_k500 / svm
  val: top1=8.28% top5=18.72% macro_f1=6.54%
  test: top1=8.82% top5=18.68% macro_f1=7.07%

## outputs/b1_full500_sift_bovw_k1000 / svm
  val: top1=8.66% top5=19.18% macro_f1=7.47%
  test: top1=8.80% top5=18.82% macro_f1=7.46%

## outputs/b1_ablate_k2000 / svm
  val: top1=9.08% top5=18.40% macro_f1=7.90%
  test: top1=8.86% top5=17.80% macro_f1=7.71%

## outputs/b1_full500_sift_bovw_k1000 / random_forest
  val: top1=2.78% top5=7.36% macro_f1=1.99%
  test: top1=2.98% top5=7.90% macro_f1=2.27%


## Appendix: mapping to the original CLI

| Notebook section | Original command |
|---|---|
| §2 Pipeline | `python -m src.traditional.run_pipeline --config ...` |
| §3 Ablation table | `python scripts/compare_vocab_ablation.py` |
| §4 NPZ export | `python scripts/export_shared_eval_scores.py` |
| §6 Cases | `python scripts/export_prediction_cases.py` |

Libraries: `src/traditional/`, `src/evaluation/`. The `scripts/*.py` files remain available as CLI backups and match this notebook's logic.
